# Album Cover Art for the Spotify Popularity Project

**Goal.** Attach an album-cover image to each song in `song_data_with_artists.csv`, so the quantile-regression model can be extended with image features (the "imaging" extension).

**What this notebook does**

1. Summarises the literature that motivates using cover art (section 1).
2. Looks up every song on **MusicBrainz** (song title + artist -> recording -> album / "release group").
3. Downloads the front cover from the **Cover Art Archive** (the image store attached to MusicBrainz).
4. Writes `song_cover_manifest.csv` (one row per song, with the path of its cover image) and shows coverage and a sample of covers.

Both steps are resumable: if the notebook is interrupted, re-running it continues where it stopped.

> **Status of this code.** The matching logic was tested offline against mocked MusicBrainz responses, but it could not be run against the live API from the environment it was written in. Run it with `LIMIT = 25` first (section 2) and check the sample output before starting the full run.

## 1. Why cover art? Literature

**The honest summary:** there is solid evidence that (a) album covers carry information about the music and (b) image content predicts popularity in other domains, and one marketing study links a cover property directly to chart performance. But published *song-popularity* models use audio, lyrics and metadata and leave the cover out. That gap is the contribution of this project: testing whether the cover adds predictive information **after** controlling for audio features and artist fame (APS), at different quantiles of popularity.

### A. Album covers carry information about the music
| Reference | What it shows | Use in our report |
|---|---|---|
| Lībeks, J. & Turnbull, D. (2011). *You Can Judge an Artist by an Album Cover: Using Images for Music Annotation.* IEEE MultiMedia 18(4), 30-37. doi:10.1109/MMUL.2011.1 | A user study and a simple vision system: cover art and promo photos predict genre tags. | The founding "covers are informative" citation. |
| Oramas, S., Nieto, O., Barbieri, F. & Serra, X. (2017). *Multi-label Music Genre Classification from Audio, Text, and Images Using Deep Features.* ISMIR 2017. arXiv:1707.04916 | A pretrained CNN on cover images predicts genre; images are weaker than audio/text alone but informative. | Precedent for our exact pipeline: pretrained network -> cover features. |
| Oramas, S., Barbieri, F., Nieto, O. & Serra, X. (2018). *Multimodal Deep Learning for Music Genre Classification.* Transactions of ISMIR 1(1), 4-21. | Combines learned representations from audio, text reviews and cover art. | Precedent for fusing cover features with other modalities. |
| *Disc-Cover Complexity Trends in Music Illustrations from Sinatra to Swift* (2025). arXiv:2510.00990 | Quantifies how cover design changed over decades and by genre. | Shows covers are a measurable cultural signal; also warns that cover style is confounded with release era. |

### B. Visual content predicts popularity and market performance
| Reference | What it shows | Use in our report |
|---|---|---|
| Joye, Y. et al. (2024). *Set that record straight! Cardinal line orientations in music album artwork boost market performance and music consumption.* Psychology & Marketing. doi:10.1002/mar.21959 | A visual property of album artwork predicts US chart position and weeks on chart. | The most direct support: cover design -> commercial outcome. |
| Khosla, A., Das Sarma, A. & Hamid, R. (2014). *What Makes an Image Popular?* WWW 2014. doi:10.1145/2566486.2567996 | Image content (colour, deep features, objects) predicts view counts, alongside social context. | Classic "image -> popularity" paper; their content-vs-social comparison mirrors our cover-vs-APS comparison. |

### C. Song-popularity prediction (where the gap is)
| Reference | What it shows | Use in our report |
|---|---|---|
| Raj, R., Wadhvani, R., Rasool, A. & Gupta, M. (2023). *Quantile Regression Approach in Analysis of Factors Affecting Song Popularity.* ICSCCC 2023. doi:10.1109/ICSCCC58608.2023.10176824 | Linear quantile regression on audio features (our baseline paper). | Baseline. |
| Yang, L.-C., Chou, S.-Y., Liu, J.-Y., Yang, Y.-H. & Chen, Y.-A. (2017). *Revisiting the Problem of Audio-based Hit Song Prediction using Convolutional Neural Networks.* ICASSP 2017. arXiv:1704.01280 | CNNs on mel-spectrograms beat shallow models for hit prediction. | Deep / image-like models already used for popularity (on audio). |
| Martín-Gutiérrez, D., Hernández Peñaloza, G., Belmonte-Hernández, A. & Álvarez García, F. (2020). *A Multimodal End-to-End Deep Learning Architecture for Music Popularity Prediction.* IEEE Access 8, 39361-39374. | HitMusicNet: audio + lyrics + metadata predict Spotify popularity (0-100). | Multimodal precedent with the same target as ours; no cover art. |
| Choudhary, Y., Rao, P. & Bhattacharyya, P. (2025). *Who Will Top the Charts? Multimodal Music Popularity Prediction via Adaptive Fusion of Modality Experts and Temporal Engagement Modeling.* arXiv:2512.06259 | Audio + lyrics + social metadata, plus artist career-trajectory features. | Recent state of the art; shows artist history matters (as our APS does); no cover art. |
| Seufitelli, D. B. et al. (2023). *Hit song science: a comprehensive survey and research directions.* Journal of New Music Research. | Survey of the field and the feature types used. | Cite for "what has been tried". |

### D. Methods we will use in the modelling notebook
| Reference | Role |
|---|---|
| Radford, A. et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). ICML 2021. | Pretrained image encoder used to turn each cover into a feature vector. |
| Selvaraju, R. R. et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.* ICCV 2017. | Heatmaps of which cover regions drive a prediction. |
| Koenker, R. & Bassett, G. (1978). *Regression Quantiles.* Econometrica 46(1), 33-50. | The check (pinball) loss. |
| Taylor, J. W. (2000). *A quantile regression neural network approach to estimating the conditional density of multiperiod returns.* Journal of Forecasting 19(4), 299-311. | Early use of the pinball loss to train a neural network (nonlinear quantile regression). |

*Check each entry against the publisher page before it goes in the reference list; page numbers and author initials above come from index pages, not the PDFs.*

## 2. Setup

**How the lookup works**

- **MusicBrainz** (`musicbrainz.org/ws/2`) is an open music encyclopedia. It has no images itself, but it gives each album a stable ID (a *release-group MBID*). We search it for a *recording* by title and artist; each result lists the releases it appears on.
- **Cover Art Archive** (`coverartarchive.org`) stores cover images keyed by those MBIDs. `/release-group/{mbid}/front-500` returns the front cover as a 500-pixel thumbnail.

**Rules to respect**

- MusicBrainz allows about **1 request per second** and requires a `User-Agent` that identifies you with a contact address. Requests without one, or sent faster, get blocked (HTTP 503). **Put your own email in `CONTACT` below.**
- No API key or login is needed for either service.

**Run time.** One MusicBrainz request per song at 1 per second: about 1.7 hours for the ~5,500 distinct songs whose artist has an APS, about 3.5 hours for all ~11,600 distinct songs with an artist name. Image downloads add roughly 20-40 minutes.

In [ ]:
import json, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image

# ---------------- configuration ----------------
CONTACT       = "your_email@example.com"     # <-- REQUIRED: your real email (MusicBrainz policy)
USER_AGENT    = f"SpotifyCoverArtClassProject/0.1 ( {CONTACT} )"

LIMIT         = 25       # number of songs to process; set to None for the full run (test with 25 first)
ONLY_APS      = True     # True = only songs whose artist is in the APS ranking (the regression sample)
IMG_SIZE      = 500      # Cover Art Archive thumbnail size: 250, 500 or 1200
MB_PAUSE      = 1.1      # seconds between MusicBrainz requests (do not go below 1.0)
DL_WORKERS    = 4        # parallel image downloads (Cover Art Archive has no stated limit; stay polite)

SONG_FILE     = "song_data_with_artists.csv"
OUT_DIR       = Path("cover_art"); OUT_DIR.mkdir(exist_ok=True)
IMG_DIR       = OUT_DIR / "images"; IMG_DIR.mkdir(exist_ok=True)
MATCH_CACHE   = OUT_DIR / "musicbrainz_matches.json"     # resumable cache of lookups
MANIFEST      = OUT_DIR / "song_cover_manifest.csv"      # final output

assert "example.com" not in CONTACT or LIMIT is not None and LIMIT <= 25, \
    "Set CONTACT to your real email before the full run."

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json"})

## 3. Songs to look up

A cover belongs to an album, so we look up each distinct (song title, primary artist) pair once. Songs with no artist name are skipped: a title alone ("Home", "Stay") matches too many different recordings to trust.

In [ ]:
def find_file(filename):
    path = Path(filename)
    if path.exists():
        return path
    for parent in Path.cwd().parents:
        if (parent / filename).exists():
            return parent / filename
    raise FileNotFoundError(filename)

songs = pd.read_csv(find_file(SONG_FILE)).drop_duplicates().reset_index(drop=True)
print("songs after dropping exact duplicates:", len(songs))

todo = songs[songs["artist_primary"].notna()].copy()
if ONLY_APS:
    todo = todo[todo["artist_streams"].notna()]          # artist_streams is present only for ranked artists
todo = (todo.groupby(["song_name", "artist_primary"], as_index=False)
            .agg(song_duration_ms=("song_duration_ms", "median")))
todo["key"] = todo["song_name"] + " ||| " + todo["artist_primary"]
if LIMIT:
    todo = todo.sample(min(LIMIT, len(todo)), random_state=0).reset_index(drop=True)

print(f"distinct (song, artist) pairs to look up: {len(todo):,}"
      f"  (~{len(todo) * MB_PAUSE / 3600:.1f} h at one request per {MB_PAUSE} s)")
todo.head()

## 4. Match each song to an album on MusicBrainz

For each song we search for recordings with that title by that artist. One song usually appears on many releases (the original album, singles, compilations, live albums, reissues). We want the cover a listener would associate with the song, so candidates are ranked by:

1. recording length close to the song's duration in our data (within 10 s),
2. official releases before bootlegs/promos,
3. plain albums/singles/EPs before compilations, live albums, remixes and soundtracks,
4. albums before singles before EPs,
5. earliest release date.

The top 3 distinct albums are kept, so if the first has no image in the Cover Art Archive we can fall back to the next.

If the full title finds nothing, we retry with version suffixes removed (`" - Radio Edit"`, `"(feat. X)"`, `"- Remastered 2011"`).

In [ ]:
MB_URL = "https://musicbrainz.org/ws/2/recording"
TYPE_RANK = {"Album": 0, "Single": 1, "EP": 2}

def phrase(s):
    # Quote a string for a Lucene phrase query.
    return '"' + str(s).replace("\\", "\\\\").replace('"', '\\"') + '"'

def title_variants(title):
    # Full title first, then progressively simplified versions.
    out = [title]
    t = re.sub(r"\s*[\(\[][^\)\]]*(feat\.?|ft\.?|with|featuring)[^\)\]]*[\)\]]", "", title, flags=re.I)
    out.append(t)
    out.append(re.split(r"\s+-\s+", t)[0])                        # drop " - Radio Edit", " - Remastered"
    out.append(re.sub(r"\s*[\(\[][^\)\]]*[\)\]]", "", out[-1]))   # drop any remaining (...) or [...]
    seen, uniq = set(), []
    for v in (x.strip() for x in out):
        if v and v.lower() not in seen:
            seen.add(v.lower()); uniq.append(v)
    return uniq

def mb_search(title, artist, retries=4):
    # One MusicBrainz recording search. Returns the list of recordings (possibly empty).
    params = {"query": f"recording:{phrase(title)} AND artist:{phrase(artist)}", "fmt": "json", "limit": 15}
    for attempt in range(retries):
        try:
            r = session.get(MB_URL, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(5 * (attempt + 1)); continue
        if r.status_code == 200:
            return r.json().get("recordings", [])
        if r.status_code in (429, 503):                           # rate-limited: back off and retry
            time.sleep(5 * (attempt + 1)); continue
        r.raise_for_status()
    raise RuntimeError(f"MusicBrainz kept refusing the request for {title!r} / {artist!r}")

def rank_candidates(recordings, duration_ms, min_score=85, keep=3):
    # Turn search results into a ranked list of candidate albums (release groups).
    cands = []
    for rec in recordings:
        if int(rec.get("score", 0)) < min_score:
            continue
        length = rec.get("length")
        dur_gap = abs(length - duration_ms) / 1000 if length and duration_ms else np.nan
        for rel in rec.get("releases", []) or []:
            rg = rel.get("release-group") or {}
            if not rg.get("id"):
                continue
            secondary = rg.get("secondary-types") or []
            cands.append({
                "rg_id": rg["id"], "release_id": rel.get("id"), "album": rg.get("title") or rel.get("title"),
                "rg_type": rg.get("primary-type"), "secondary": "/".join(secondary),
                "status": rel.get("status"), "date": rel.get("date") or "", "dur_gap_s": dur_gap,
                "sort": (0 if (dur_gap == dur_gap and dur_gap <= 10) else 1,
                         0 if rel.get("status") == "Official" else 1,
                         1 if secondary else 0,
                         TYPE_RANK.get(rg.get("primary-type"), 3),
                         rel.get("date") or "9999"),
            })
    cands.sort(key=lambda c: c["sort"])
    best, seen = [], set()
    for c in cands:                                               # keep the best release of each album
        if c["rg_id"] not in seen:
            seen.add(c["rg_id"]); c.pop("sort"); best.append(c)
        if len(best) == keep:
            break
    return best

def lookup(title, artist, duration_ms):
    for variant in title_variants(title):
        recs = mb_search(variant, artist)
        time.sleep(MB_PAUSE)
        cands = rank_candidates(recs, duration_ms)
        if cands:
            return {"query_title": variant, "candidates": cands}
    return {"query_title": None, "candidates": []}

In [ ]:
# Resumable loop: results are saved every 25 songs, and songs already in the cache are skipped.
matches = json.loads(MATCH_CACHE.read_text()) if MATCH_CACHE.exists() else {}
pending = todo[~todo["key"].isin(matches)]
print(f"already looked up: {len(todo) - len(pending):,} | to do now: {len(pending):,}")

t0 = time.time()
for i, row in enumerate(pending.itertuples(index=False), 1):
    try:
        matches[row.key] = lookup(row.song_name, row.artist_primary, row.song_duration_ms)
    except Exception as e:                                        # keep going; failed songs are retried on re-run
        print(f"  skipped {row.key!r}: {e}")
    if i % 25 == 0 or i == len(pending):
        MATCH_CACHE.write_text(json.dumps(matches))
        found = sum(bool(matches[k]["candidates"]) for k in todo["key"] if k in matches)
        print(f"{i:>6,}/{len(pending):,}  matched so far: {found:,}  elapsed: {(time.time() - t0) / 60:.1f} min")

## 5. Download the covers from the Cover Art Archive

Images are saved once per album as `images/{release-group MBID}.jpg`, so songs from the same album share one file. For each song we try its candidate albums in order and stop at the first one that has a front cover. HTTP 404 means the archive has no front image for that album.

In [ ]:
CAA_URL = "https://coverartarchive.org/release-group/{mbid}/front-{size}"

import io, threading
from collections import defaultdict

_locks = defaultdict(threading.Lock)      # one lock per album, so two threads never fetch the same cover
_locks_guard = threading.Lock()

def fetch_cover(rg_id, retries=3):
    # Download one album cover. Returns 'ok', 'cached', 'missing' (no art exists) or 'error'.
    path = IMG_DIR / f"{rg_id}.jpg"
    miss = IMG_DIR / f"{rg_id}.missing"
    with _locks_guard:
        lock = _locks[rg_id]
    with lock:
        if path.exists(): return "cached"
        if miss.exists(): return "missing"
        for attempt in range(retries):
            try:
                r = requests.get(CAA_URL.format(mbid=rg_id, size=IMG_SIZE),
                                 headers={"User-Agent": USER_AGENT}, timeout=60)   # redirects to archive.org
            except requests.RequestException:
                time.sleep(3 * (attempt + 1)); continue
            if r.status_code == 200:
                try:
                    with Image.open(io.BytesIO(r.content)) as im:      # decode in memory, no temp file (Windows-safe)
                        im.convert("RGB").save(path, "JPEG", quality=92)
                    return "ok"
                except Exception:
                    return "error"
            if r.status_code == 404:
                miss.touch(); return "missing"
            time.sleep(3 * (attempt + 1))                          # 429 / 5xx: wait and retry
        return "error"

def cover_for_song(key):
    # Try the song's candidate albums in order; return the first that has a cover.
    for rank, c in enumerate(matches.get(key, {}).get("candidates", [])):
        if fetch_cover(c["rg_id"]) in ("ok", "cached"):
            return {**c, "candidate_rank": rank, "cover_path": str(IMG_DIR / f"{c['rg_id']}.jpg")}
    return None

keys = list(todo["key"])
t0 = time.time()
with ThreadPoolExecutor(DL_WORKERS) as pool:
    chosen = list(pool.map(cover_for_song, keys))
print(f"downloads finished in {(time.time() - t0) / 60:.1f} min; "
      f"{len(list(IMG_DIR.glob('*.jpg'))):,} cover files on disk")

## 6. Manifest and coverage

`song_cover_manifest.csv` has one row per (song, artist) with the matched album, how it was matched, and the image path. Merge it back onto the song data on `song_name` + `artist_primary`.

In [ ]:
rows = []
for key, row, pick in zip(keys, todo.itertuples(index=False), chosen):
    m = matches.get(key, {})
    base = {"song_name": row.song_name, "artist_primary": row.artist_primary,
            "query_title": m.get("query_title"), "n_candidates": len(m.get("candidates", []))}
    rows.append({**base, **(pick or {})})
manifest = pd.DataFrame(rows)
for col in ["rg_id", "cover_path", "album", "rg_type", "secondary", "status", "date", "dur_gap_s", "candidate_rank"]:
    if col not in manifest: manifest[col] = np.nan
manifest["has_cover"] = manifest["cover_path"].notna()
manifest.to_csv(MANIFEST, index=False)

n = len(manifest)
print(f"songs looked up            : {n:,}")
print(f"matched on MusicBrainz     : {(manifest['n_candidates'] > 0).sum():,} ({(manifest['n_candidates'] > 0).mean():.1%})")
print(f"with a downloaded cover    : {manifest['has_cover'].sum():,} ({manifest['has_cover'].mean():.1%})")
print(f"distinct covers            : {manifest['rg_id'].nunique():,}")
print(f"matched with duration <=10s: {(manifest['dur_gap_s'] <= 10).sum():,}")
print("\nalbum type of the chosen cover:")
print(manifest.loc[manifest["has_cover"], "rg_type"].fillna("unknown").value_counts().to_string())
manifest.head(10)

### Visual check

Look at a random sample and confirm the covers belong to the right songs. Typical errors to look for: a compilation cover ("Now That's What I Call Music"), a same-titled song by a different act, or a live album.

In [ ]:
sample = manifest[manifest["has_cover"]].sample(min(24, int(manifest["has_cover"].sum())), random_state=1)
if len(sample):
    cols = 6; nrows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(cols * 2.6, nrows * 3.1))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, r in zip(axes, sample.itertuples(index=False)):
        ax.imshow(Image.open(r.cover_path))
        ax.set_title(f"{str(r.song_name)[:24]}\n{str(r.artist_primary)[:24]}", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "cover_sample.png", dpi=150); plt.show()
else:
    print("No covers downloaded yet.")

In [ ]:
# Songs that still have no cover (for a fallback source or manual inspection)
missing = manifest[~manifest["has_cover"]]
print(f"{len(missing):,} songs without a cover; "
      f"{(missing['n_candidates'] == 0).sum():,} had no MusicBrainz match, "
      f"{(missing['n_candidates'] > 0).sum():,} matched but the archive has no image.")
missing[["song_name", "artist_primary", "n_candidates"]].head(20)

## 7. Notes for the modelling step

- **Split by artist, and never split an album.** Many songs share one cover (same `rg_id`). A random train/test split would let the model see a test cover during training. Keep using `GroupKFold` on the artist, as in the quantile-regression notebook.
- **Selection.** Covers exist only for songs matched here. Report coverage, and check whether matched songs differ in popularity from unmatched ones before interpreting results.
- **Cover is album-level, popularity is song-level.** All songs on an album share a cover but have different popularity, which caps how much a cover can explain. Say so in the report.
- **Release era is a confounder.** Cover style changes over time and Spotify popularity favours recent plays. The manifest's `date` column lets you add release year as a control.
- **Copyright.** Cover images are copyrighted by the labels. Using them for class research is routine; do not redistribute the image folder publicly (share the manifest with MBIDs instead).
- **Attribution.** Credit "MusicBrainz" and "Cover Art Archive" as data sources in the report.